# Beam Search & BLEU Score

> Based on Stanford CS230 — C5M3, Andrew Ng / DeepLearning.AI

Greedy decoding picks the single best token at each step, which can miss globally better sequences. **Beam search** maintains the $B$ most probable partial sequences at each step. **BLEU score** evaluates generated text against reference translations.

## Learning Objectives

1. Implement beam search and explain the beam width trade-off
2. Understand length normalisation to prevent short-output bias
3. Compute modified n-gram precision for BLEU
4. Apply the brevity penalty and compute the final BLEU score

## Beam Search

At each step keep the top $B$ partial sequences by **cumulative log-probability**:

$$\text{score}(y^{\langle 1 \rangle}, \ldots, y^{\langle t \rangle}) = \frac{1}{T_y^\alpha}\sum_{t=1}^{T_y} \log P\!\bigl(y^{\langle t \rangle} \mid x, y^{\langle 1 \rangle},\ldots,y^{\langle t-1 \rangle}\bigr)$$

Length normalisation exponent $\alpha \in [0,1]$ prevents penalising longer outputs too heavily.

## BLEU Score

For $n$-gram precision $p_n$ across candidate and reference:

$$p_n = \frac{\sum_{\text{n-gram} \in \hat{y}} \text{count\_clip}(\text{n-gram})}{\sum_{\text{n-gram} \in \hat{y}} \text{count}(\text{n-gram})}$$

$$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \le r \end{cases}$$

$$\text{BLEU} = \text{BP} \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right), \quad w_n = \frac{1}{N}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- BLEU implementation ----
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def modified_precision(candidate, references, n):
    cand_ngrams = Counter(ngrams(candidate, n))
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = Counter(ngrams(ref, n))
        for ng in cand_ngrams:
            max_ref_counts[ng] = max(max_ref_counts[ng], ref_ngrams[ng])
    clipped = {ng: min(cnt, max_ref_counts[ng]) for ng, cnt in cand_ngrams.items()}
    denom = max(1, len(candidate) - n + 1)
    return sum(clipped.values()) / denom

def brevity_penalty(candidate, references):
    c = len(candidate)
    r = min(len(ref) for ref in references, key=lambda ref: abs(len(ref)-c))
    return 1.0 if c > r else np.exp(1 - r/c)

def bleu(candidate, references, N=4):
    bp = brevity_penalty(candidate, references)
    log_avg = 0
    for n in range(1, N+1):
        pn = modified_precision(candidate, references, n)
        log_avg += (1/N) * np.log(pn + 1e-10)
    return bp * np.exp(log_avg)

# Example from CS230 lecture
ref1 = "the cat is on the mat".split()
ref2 = "there is a cat on the mat".split()

cand_good = "the cat is on the mat".split()
cand_ok   = "the cat sat on the mat".split()
cand_bad  = "the the the the the the".split()

references = [ref1, ref2]
for label, cand in [("Perfect match", cand_good), ("One word off", cand_ok), ("Degenerate", cand_bad)]:
    score = bleu(cand, references)
    print(f"{label:20s}  BLEU = {score:.4f}   candidate: '{' '.join(cand)}'")

# ---- Effect of beam width (simulated) ----
np.random.seed(7)
beam_widths = [1, 2, 3, 5, 10, 20]
# Simulate: wider beam → better score (diminishing returns after ~5)
base_bleu = 0.30
quality = [base_bleu + 0.12 * (1 - 1/(0.3*b + 1)) + 0.01*np.random.randn()
           for b in beam_widths]
compute  = [b**1.6 for b in beam_widths]  # roughly super-linear cost

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Beam Search — Width vs Quality & Compute', fontsize=13, fontweight='bold')

axes[0].plot(beam_widths, quality, 'o-', color=P[3], lw=2, ms=8)
axes[0].set_xlabel('Beam width B')
axes[0].set_ylabel('BLEU score (simulated)')
axes[0].set_title('Translation quality vs beam width')
axes[0].grid(True)
axes[0].set_xticks(beam_widths)

axes[1].plot(beam_widths, compute, 's-', color=P[1], lw=2, ms=8)
axes[1].set_xlabel('Beam width B')
axes[1].set_ylabel('Relative compute (a.u.)')
axes[1].set_title('Compute cost vs beam width (~B^1.6)')
axes[1].grid(True)
axes[1].set_xticks(beam_widths)

plt.tight_layout()
plt.show()

# ---- BLEU n-gram precision breakdown ----
cands = {'Perfect': cand_good, 'One-off': cand_ok, 'Degenerate': cand_bad}
ns    = [1, 2, 3, 4]
fig, ax = plt.subplots(figsize=(10, 5))
w = 0.25
for j, (label, cand) in enumerate(cands.items()):
    pns = [modified_precision(cand, references, n) for n in ns]
    ax.bar(np.array(ns) + (j-1)*w, pns, width=w, color=P[j], alpha=0.85, label=label)
ax.set_xlabel('n-gram order'); ax.set_ylabel('Modified precision')
ax.set_title('BLEU n-gram Precision Breakdown')
ax.set_xticks(ns); ax.set_xticklabels([f'{n}-gram' for n in ns])
ax.legend(); ax.grid(True, axis='y')
plt.tight_layout()
plt.show()
